# 🚗 차량 인스턴스 세그멘테이션 — 학습·평가 노트북 (Colab)

**Segmentation 기반 도로 CCTV 의 도로 대비 차량 면적 비를 활용한 교통 혼잡도 측정 시스템**

이 노트북은 `training/` 폴더의 코드를 **순서대로 호출만** 한다. 실제 로직은 전부 `.py` 파일에 있다.

| 단계 | 셀 | 하는 일 |
|---|---|---|
| 0 | 환경 | GPU 확인 · 드라이브 마운트 · 코드 준비 · 데이터 풀기 · 설치 |
| 1 | 점검 | 이미지 몇십 장으로 전체 흐름이 도는지 확인 (몇 분) |
| 2 | 사전학습 평가 | COCO 사전학습 모델을 **학습 없이** val 로 평가 → 기준선 |
| 3 | 파인튜닝 | 모델별 학습 → val 채점 → 기록표 자동 추가 |
| 4 | 실험 반복 | 하이퍼파라미터를 바꿔 가며 여러 번 학습 |
| 5 | 기록표 | `experiments.csv` → 표 |
| 6 | 최종 test | 고른 모델만 test 로 한 번 |
| 7 | 추론 | 차량 대수 · 픽셀 수 · 합집합 마스크 · 시각화 |

> 결과(`runs/`)는 **드라이브**에 저장된다. 세션이 끊겨도 기록표·가중치가 남고 `train.resume` 으로 이어서 학습할 수 있다.

## 0. 환경 준비
런타임 → 런타임 유형 변경 → **GPU** (A100/L4 권장, T4 는 batch 를 줄일 것)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ===== 경로 설정 — 본인 드라이브 구조에 맞게 수정 =====
DRIVE_DIR = "/content/drive/MyDrive/carseg"                 # 프로젝트 폴더
TAR_PATH  = f"{DRIVE_DIR}/car_seg_split_upload.tar"          # 업로드한 데이터 tar
RUNS_DIR  = f"{DRIVE_DIR}/runs"                               # 학습 결과·기록표 저장 위치 (드라이브)

# 코드 가져오는 방법: "git" (GitHub 저장소) 또는 "drive" (드라이브에 training 폴더 업로드)
CODE_SOURCE = "git"
GIT_URL     = "https://github.com/Hanhws/Machine-Learning-Deep-Learning-Team-Project.git"
DRIVE_CODE  = f"{DRIVE_DIR}/training"

DATA_ROOT = "/content/car_seg_split_upload"                  # 코랩 로컬 디스크 (빠름)
CODE_DIR  = "/content/training"

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, subprocess
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)
if CODE_SOURCE == "git":
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, "/content/repo"], check=True)
    shutil.copytree("/content/repo/training", CODE_DIR)
else:
    shutil.copytree(DRIVE_CODE, CODE_DIR)
os.chdir(CODE_DIR)
print("코드:", os.getcwd(), os.listdir())

In [ ]:
# 데이터 tar 를 코랩 로컬 디스크에 푼다 (드라이브에서 직접 읽으면 매우 느림). 약 5~10분.
import os
if not os.path.exists(f"{DATA_ROOT}/annotations/instances_val.json"):
    !tar -xf "{TAR_PATH}" -C /content
    !find /content -maxdepth 4 -name '._*' -delete   # macOS 가 만든 ._ 파일 정리

# 검증 — 기대값: train 20463 / val 2560 / test 2566
for s in ["train", "val", "test"]:
    n_img = len(os.listdir(f"{DATA_ROOT}/images/{s}"))
    n_msk = len(os.listdir(f"{DATA_ROOT}/masks/{s}"))
    print(f"{s:5s} images={n_img:6d} masks={n_msk:6d} coco={os.path.exists(f'{DATA_ROOT}/annotations/instances_{s}.json')}")

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# 모든 명령에 공통으로 붙일 덮어쓰기
COMMON = f"data.root={DATA_ROOT} out_dir={RUNS_DIR}"
os.makedirs(RUNS_DIR, exist_ok=True)

## 1. 빠른 점검 (선택, 권장)
이미지 32장 · 1 epoch 로 **학습 → 채점 → 기록** 이 끝까지 도는지 확인. 결과는 `runs_smoke/` 에 따로 저장된다.

In [ ]:
SMOKE = f"data.root={DATA_ROOT} out_dir=/content/runs_smoke data.train_limit=32 data.eval_limit=16 train.epochs=1"
!python train.py --config configs/yolo11s_seg.yaml    --set {SMOKE} train.batch=4 train.imgsz=640
!python train.py --config configs/maskrcnn_r50_v2.yaml --set {SMOKE} train.batch=2 eval.eval_every=1

## 2. 사전학습 모델 그대로 평가 (zero-shot 기준선)
COCO 80종으로 학습된 모델에서 **car / bus / truck** 예측만 추려 우리 val 정답과 비교한다.
파인튜닝이 얼마나 도움이 됐는지 비교하는 기준선이 된다. 기록표에 `stage=pretrained` 로 남는다.

In [ ]:
!python evaluate.py --pretrained --set {COMMON} --config \
    configs/yolov8s_seg.yaml configs/yolo11s_seg.yaml configs/yolo26s_seg.yaml \
    configs/maskrcnn_r50_v2.yaml configs/mask2former_swin_s.yaml configs/rfdetr_seg_medium.yaml

## 3. 파인튜닝 — 모델별
각 셀이 독립적이다. 학습이 끝나면 **best 가중치로 val 을 통합 채점**하고 기록표에 한 행을 추가한다.

- 실험 이름(`name`)이 같으면 같은 폴더를 덮어쓰니, 설정을 바꿀 때마다 이름도 바꾼다.
- 세션이 끊기면: `--set ... train.resume=<RUNS_DIR>/<name>/.../last.pt` (아래 표 참고)

| 계열 | 이어하기 체크포인트 |
|---|---|
| YOLO | `<name>/ultralytics/weights/last.pt` |
| Mask R-CNN · Mask2Former | `<name>/weights/last.pt` |
| RF-DETR | `<name>/rfdetr/last.ckpt` |

### 3-1. YOLO (v8 / 11 / 26)

In [ ]:
!python train.py --config configs/yolo11s_seg.yaml --set {COMMON}

In [ ]:
!python train.py --config configs/yolov8s_seg.yaml --set {COMMON}

In [ ]:
!python train.py --config configs/yolo26s_seg.yaml --set {COMMON}

### 3-2. Mask R-CNN

In [ ]:
!python train.py --config configs/maskrcnn_r50_v2.yaml --set {COMMON}

### 3-3. Mask2Former

In [ ]:
!python train.py --config configs/mask2former_swin_s.yaml --set {COMMON}

### 3-4. RF-DETR Segmentation

In [ ]:
!python train.py --config configs/rfdetr_seg_medium.yaml --set {COMMON}

## 4. 실험 반복 (하이퍼파라미터 탐색)
성능기록표처럼 **한 번에 한두 가지만 바꿔** 효과를 본다. 아래는 예시 — 원하는 조합으로 바꿔 쓴다.
`group` 으로 묶어 두면 기록표에서 그룹별로 비교된다.

In [ ]:
experiments = [
    # (설정 파일, 실험 이름, 그룹, 바꿀 값)
    ("configs/yolo11s_seg.yaml", "yolo11s_img960",   "YOLO imgsz", "train.imgsz=960"),
    ("configs/yolo11s_seg.yaml", "yolo11s_img1536",  "YOLO imgsz", "train.imgsz=1536 train.batch=8"),
    ("configs/yolo11s_seg.yaml", "yolo11m_img1280",  "YOLO size",  "model.weights=yolo11m-seg.pt train.batch=8"),
    ("configs/yolo11s_seg.yaml", "yolo11s_nomosaic", "YOLO aug",   "train.mosaic=0"),
    ("configs/yolo11s_seg.yaml", "yolo11s_cp0.3",    "YOLO aug",   "train.copy_paste=0.3"),
]
for cfg, name, group, sets in experiments:
    print("=" * 80, "\n", name, "|", sets)
    !python train.py --config {cfg} --set {COMMON} name={name} group="{group}" notes="{sets}" {sets}

### 4-1. 추론 설정만 바꿔 다시 채점 (학습 없이)
임계값·TTA 는 재학습 없이 평가만 다시 하면 된다 (성능기록표의 *Thrsh 조정* 그룹).

In [ ]:
run = f"{RUNS_DIR}/yolo11s_seg_1280"
!python evaluate.py --run {run} --set eval.conf=0.001 group="Thrsh 조정" notes="conf=0.001"
!python evaluate.py --run {run} --set eval.tta=true group="Thrsh 조정" notes="TTA"

## 5. 성능 기록표

In [ ]:
!python make_report.py --csv {RUNS_DIR}/experiments.csv --split val

import pandas as pd
df = pd.read_csv(f"{RUNS_DIR}/experiments.csv")
cols = ["no", "group", "exp", "stage", "split", "weights", "imgsz", "batch", "epochs_done", "lr0",
        "mask_mAP", "mask_mAP50", "mask_mAP_small", "box_mAP", "AP_car", "AP_bus", "AP_truck",
        "vehicle_IoU", "area_ratio_MAE_pp", "fps", "train_time", "notes"]
df[df.split == "val"][cols].sort_values("mask_mAP", ascending=False)

## 6. 최종 test 평가
**val 로 모델을 다 고른 뒤, 최종 후보만** test 로 한 번 평가한다 (test 를 보고 설정을 고르면 과대평가가 된다).

In [ ]:
FINAL_RUNS = [f"{RUNS_DIR}/yolo11s_seg_1280"]   # ← 최종 후보 run 폴더들
!python evaluate.py --split test --run {" ".join(FINAL_RUNS)}
!python make_report.py --csv {RUNS_DIR}/experiments.csv --split test

## 7. 추론 — 차량 대수 · 픽셀 수 · 합집합 마스크
- 차량 픽셀 = 인스턴스 마스크 **합집합** (`masks.any(axis=0)`) → 겹친 픽셀은 한 번만
- `--road-mask` 로 도로 마스크(PNG, 0=배경)를 주면 **도로 대비 점유율** 까지 계산

In [ ]:
RUN = f"{RUNS_DIR}/yolo11s_seg_1280"
!python infer.py --run {RUN} --source {DATA_ROOT}/images/test --limit 20 --save-vis --save-masks
pd.read_csv(f"{RUN}/infer/results.csv").head()

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{RUN}/infer/vis/*.jpg"))[:3]:
    display(Image(p, width=960))

### 7-1. 파이썬에서 직접 쓰기 (혼잡도 시스템 연동용)

In [ ]:
import sys, numpy as np
sys.path.insert(0, CODE_DIR)
from carseg.inference import VehicleSegmenter

seg = VehicleSegmenter.from_run(RUN)                 # config.yaml + best 가중치 자동
img_path = sorted(glob.glob(f"{DATA_ROOT}/images/test/*.jpg"))[0]
out = seg.run(img_path)                              # road_mask=도로모델출력(bool HxW) 을 주면 점유율 계산

masks = out.prediction.above(seg.score_thr).masks   # (N, H, W) 인스턴스 마스크
union_mask = masks.any(axis=0)                       # 합집합
print("차량 수      :", out.stats.num_vehicles, out.stats.counts)
print("차량 픽셀 수 :", int(union_mask.sum()), "(= stats.vehicle_pixels", out.stats.vehicle_pixels, ")")
print("겹쳐 제외된 픽셀:", out.stats.overlap_pixels)
print("이미지 대비 면적비:", f"{out.stats.image_ratio*100:.2f}%")